In [14]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [18]:
df = pd.read_parquet("base_EPEX_df.parquet")[["Hour", "FORECASTDEMAND", "Incentive_forecast", "Price (GBP/MWhe)"]]

In [19]:
target = "Price (GBP/MWhe)"

# --- Reshape from hourly rows to one row per day ---
d = df.sort_values("Hour").copy()
d["date"] = pd.to_datetime(d["Hour"].dt.date)   # naive local calendar date
d["hod"] = d["Hour"].dt.hour

# Keep only complete 24-hour days (drops DST days with 23/25 hours and partial days)
d = d[d.groupby("date")["Hour"].transform("size") == 24]

def wide(col, prefix):
    w = d.pivot(index="date", columns="hod", values=col)
    w.columns = [f"{prefix}_h{h:02d}" for h in w.columns]
    return w

demand = wide("FORECASTDEMAND", "demand")
wind = wide("Incentive_forecast", "wind")
price = wide(target, "price")

# Autoregressive features: shift(freq="D") shifts by true calendar days,
# so missing days produce NaN rather than silently misaligned rows
price_lag1 = price.shift(1, freq="D").add_suffix("_lag1d")   # yesterday's 24 prices
price_lag7 = price.shift(7, freq="D").add_suffix("_lag7d")   # same weekday last week

X = (demand.join(wind, how="inner")
           .join(price_lag1, how="inner")
           .join(price_lag7, how="inner"))
X["day_of_week"] = X.index.dayofweek
X = X.dropna()
Y = price.loc[X.index]    # 24 columns: the target vector for each day

numeric_features = [c for c in X.columns if c != "day_of_week"]

# --- Pipeline ---
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), ["day_of_week"]),
])

model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression()),   # multi-output: fits one linear model per hour
])

# --- Time-ordered cross-validation ---
tscv = TimeSeriesSplit(n_splits=5)
rmses = []
for train_idx, test_idx in tscv.split(X):
    model.fit(X.iloc[train_idx], Y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    rmses.append(np.sqrt(np.mean((Y.iloc[test_idx].values - pred) ** 2)))
print("CV RMSE per fold:", np.round(rmses, 2), "mean:", np.mean(rmses).round(2))

# --- Hold-out evaluation (last 20% of days) ---
split = int(len(X) * 0.8)
model.fit(X.iloc[:split], Y.iloc[:split])
pred = pd.DataFrame(model.predict(X.iloc[split:]), index=Y.index[split:], columns=Y.columns)

err = pred - Y.iloc[split:]
print("Overall MAE :", err.abs().values.mean())
print("Overall RMSE:", np.sqrt((err ** 2).values.mean()))
print("MAE by hour:\n", err.abs().mean().round(2))

CV RMSE per fold: [24.7  69.44 37.33 24.98 19.19] mean: 35.13
Overall MAE : 12.985985777793438
Overall RMSE: 19.059575338438535
MAE by hour:
 price_h00     3.90
price_h01     4.83
price_h02     6.45
price_h03     7.67
price_h04     8.70
price_h05    10.30
price_h06    11.39
price_h07    11.87
price_h08    13.81
price_h09    14.87
price_h10    14.62
price_h11    15.01
price_h12    15.66
price_h13    15.62
price_h14    15.23
price_h15    14.48
price_h16    17.29
price_h17    26.80
price_h18    21.00
price_h19    16.93
price_h20    12.14
price_h21    10.01
price_h22    10.43
price_h23    12.64
dtype: float64


In [20]:
def metrics(y_true, y_pred):
    err = np.asarray(y_pred) - np.asarray(y_true)
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean())}

Y_test = Y.iloc[split:]
naive_prev_day = X.iloc[split:][[f"{c}_lag1d" for c in Y.columns]].values

results = pd.DataFrame({
    "Regression": metrics(Y_test, pred),
    "Prev day":   metrics(Y_test, naive_prev_day),
}).T
print(results.round(2))

# Per-hour MAE
by_hour = pd.DataFrame({
    "Regression": (pred - Y_test).abs().mean().values,
    "Prev day":   np.abs(naive_prev_day - Y_test.values).mean(axis=0),
}, index=range(24))
print(by_hour.round(2))

# Improvement over baseline (positive = regression is better)
skill = 1 - results.loc["Regression", "RMSE"] / results.loc["Prev day", "RMSE"]
print(f"RMSE improvement vs prev-day baseline: {skill:.1%}")

              MAE   RMSE
Regression  12.99  19.06
Prev day    17.49  28.50
    Regression  Prev day
0         3.90     14.83
1         4.83     14.74
2         6.45     14.89
3         7.67     14.95
4         8.70     15.35
5        10.30     17.02
6        11.39     19.16
7        11.87     21.07
8        13.81     21.55
9        14.87     21.80
10       14.62     21.92
11       15.01     22.26
12       15.66     22.70
13       15.62     22.42
14       15.23     21.64
15       14.48     20.38
16       17.29     17.90
17       26.80     15.84
18       21.00     15.79
19       16.93     14.16
20       12.14     12.06
21       10.01     11.01
22       10.43     11.80
23       12.64     14.60
RMSE improvement vs prev-day baseline: 33.1%
